# Noise level under the measured protocol

**Why this exists.** The brain training range `NOISE_STD = [0.04, 0.06]` (val 0.05)
was picked by eye on the *legacy* protocol: 20 ACS lines added on top of the
outer lines (R nominal), stored ESPIRiT maps in the operator, synthetic k-space.
The measured protocol (2026-09-22) changed three things at once, and all three
make the problem harder at the same `sigma`:

| | legacy | measured |
|---|---|---|
| ACS | 20 lines, on top of the outer lines | `center_frac=0.04` -> 13 lines, **inside** the budget (`adjust_accel`) |
| effective R at nominal 4 / 8 / 16 (N=320) | 3.4 / 5.6 / 8.2 | 4.3 / 7.8 / **15.2** |
| operator maps | stored, from fully sampled data | estimated online from the 13-line ACS of the noisy measurement |
| k-space | `M F S x` + AWGN | measured k-space (scanner noise included) + AWGN |

At R=16 the measured mask keeps **21 of 320 lines, 13 of them ACS**. The old R=16
arm was really R~8, so a sigma that was "recognizable but noisy" there is now
being stacked on twice the undersampling and on maps calibrated from 13 noisy
lines.

**What this notebook does.**

1. Masks under both protocols: lines kept, effective R.
2. The scanner's own noise floor `sigma_0` (measured k-space already has it).
3. CG-SENSE reconstructions over a sigma grid, measured protocol, every R --
   the usual criterion: pick the level where the recon is recognizable but
   fairly noisy. Legacy runs alongside, so the old operating point (R, 0.05) is
   on the same axes, and a **difficulty-matched** sigma is read off: the
   measured-protocol sigma with the same NRMSE as legacy at 0.05.
4. The coil maps estimated from the undersampled, noisy k-space (Walsh and
   ESPIRiT) against the stored ones.
5. The coil images behind them: fully sampled, the windowed ACS images Walsh
   actually fits, and the aliased measurement.

Everything goes through the training code itself -- `get_mask_cached`,
`prepare_measurement`, `online_smaps`, `compute_metrics` -- so what is shown is
what the nets see.

In [ ]:
import json
import math
import os
import pathlib
import sys

ROOT = pathlib.Path('/scratch/ee2178/ImMAP')
if not ROOT.is_dir():                                   # not on the cluster
    ROOT = pathlib.Path.cwd()
    if not (ROOT / 'physics').is_dir():
        ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch

from operators import FFT2D, Mask, Sense
from operators.fourier import fftc, ifftc
from physics.mask import get_mask_cached as get_mask, resolve_acs_lines
from physics.online_smaps import center_mask, hamming_window, online_smaps
from solvers.cg import cg
from training.common import prepare_measurement
from training.metrics import compute_metrics

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"root={ROOT}  device={device}")

## Knobs

`CONFIG` is read for **data paths and `scale_fac` only** -- the protocol is set
here explicitly (`NEW`, `OLD`), so this works whichever protocol the config on
disk was generated with.

`SMAP_METHOD` is the estimator the *operator* uses in the sweep. Walsh is the
planned switch (Sljiva's multigrid runs use it); set `'espirit'` to see what the
current configs train on.

CG-SENSE is unregularized by default and stopped at `CG_ITERS`: at R=16 the
normal operator is badly conditioned and CG semi-converges, so the iteration
count *is* the regularization. Keep it fixed across the sweep, or the columns
stop being comparable.

In [ ]:
# ============================ KNOBS ============================
SOURCE      = 'fastmri'      # 'phantom': self-contained, no data (runs anywhere)
CONFIG      = 'config/brain/mg/lpdsnet_R16.json'
SPLIT       = 'val'
N_SLICES    = 4              # metrics are averaged over these; figures use the first
R_LIST      = [4, 8, 16]
SIGMA_GRID  = [0.0, 0.005, 0.01, 0.015, 0.02, 0.03, 0.04, 0.05]
SMAP_METHOD = 'walsh'        # operator maps in the sweep: 'walsh' | 'espirit'
CG_ITERS    = 30
CG_TOL      = 1e-6
LAMDA       = 0.0            # Tikhonov weight in (E^H E + LAMDA I) x = E^H y
SEED        = 0              # one noise draw, SCALED across the grid -> columns differ only in sigma

NEW = dict(acs_lines=None, center_frac=0.04, adjust_accel=True)   # measured protocol
OLD = dict(acs_lines=20, center_frac=None, adjust_accel=False)    # legacy
OLD_SIGMA = 0.05             # the brain val sigma, chosen under OLD

SHOW_R, SHOW_SIGMA = 16, 0.02     # sections 4-5
N_COILS_SHOW    = 8
INCLUDE_ESPIRIT = True            # online ESPIRiT is GBs per slice at 640x320x20
# ===============================================================

## 0) Data

The organ mask is the support of the **stored** ESPIRiT maps
(`organ_mask_source='smaps'`), the region the masked metrics will use. Metrics
are reported both masked and unmasked.

`'phantom'` builds a birdcage-coil ellipse phantom with a known scanner noise
`sigma_0`, so the notebook can be checked end to end without data.

In [ ]:
def load_fastmri(n):
    cfg = json.load(open(CONFIG))
    d = dict(cfg['data'][SPLIT])
    d['batch_size'] = 1
    d['organ_mask_source'] = 'smaps'
    from datasets.registry import build_loader
    loader = build_loader(d, shuffle=False, drop_last=False)
    out = []
    for _, (k, s, x, om, _pad) in zip(range(n), loader):
        if k.dim() == 3:
            k, s, x, om = k[None], s[None], x[None], om[None]
        out.append(dict(kspace=k.to(device).to(torch.complex64),
                        smaps=s.to(device).to(torch.complex64),
                        image=x.to(device).to(torch.complex64),
                        organ=om.to(device).bool()))
    print(f"{CONFIG} [{SPLIT}]  scale_fac={d.get('scale_fac')}  "
          f"smap_root={d.get('smap_root')}")
    return out


def load_phantom(n, H=192, W=192, C=8, sigma0=0.004):
    yy = torch.linspace(-1, 1, H)[:, None]
    xx = torch.linspace(-1, 1, W)[None, :]
    obj = torch.zeros(H, W)
    for cy, cx, ry, rx, v in [(0, 0, .85, .70, 1.0), (-.10, 0, .55, .45, .55),
                              (.25, -.30, .18, .12, 1.3), (-.35, .28, .22, .15, .25)]:
        obj = torch.where(((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 <= 1,
                          torch.full_like(obj, v), obj)
    body = obj > 0
    s = []
    for c in range(C):
        th = 2 * math.pi * c / C
        py, px = 1.35 * math.cos(th), 1.35 * math.sin(th)
        amp = 1.0 / ((yy - py) ** 2 + (xx - px) ** 2 + 0.25)
        s.append(amp * torch.exp(1j * 2.2 * (yy * py + xx * px)))
    s = torch.stack(s)[None].to(torch.complex64)
    s = s / s.abs().pow(2).sum(1, keepdim=True).sqrt()
    s = torch.where(body, s, torch.zeros_like(s))          # stored maps: hard support
    out = []
    for i in range(n):
        g = torch.Generator().manual_seed(100 + i)
        x = (obj * torch.exp(1j * 0.6 * yy * (i + 1)))[None, None].to(torch.complex64)
        k = fftc(s * x) + sigma0 * torch.randn(s.shape, dtype=torch.complex64, generator=g)
        out.append(dict(kspace=k.to(device), smaps=s.to(device), image=x.to(device),
                        organ=body[None, None].to(device)))
    print(f"phantom {H}x{W}, {C} coils, scanner sigma_0 = {sigma0}")
    return out


batches = load_fastmri(N_SLICES) if SOURCE == 'fastmri' else load_phantom(N_SLICES)
b0 = batches[0]
_, C, H, W = b0['kspace'].shape
print(f"{len(batches)} slices   kspace {tuple(b0['kspace'].shape)}   "
      f"image {tuple(b0['image'].shape)}   organ fraction {b0['organ'].float().mean():.2f}")

VMAX = float(b0['image'].abs().max())    # ONE display window for every image panel

## 1) The masks

`lines` counts sampled phase-encode columns; `A_eff = N / lines` is the rate the
reconstruction actually faces.

In [ ]:
def lines_of(m):
    return int((m.reshape(-1, m.shape[-1]).amax(0) > 0).sum())


print(f"{'R':>3} | {'OLD lines':>9} {'A_eff':>6} {'acs':>4} | "
      f"{'NEW lines':>9} {'A_eff':>6} {'acs':>4}")
fig, ax = plt.subplots(2, len(R_LIST), figsize=(3.2 * len(R_LIST), 4.6))
for j, R in enumerate(R_LIST):
    row = []
    for i, (name, p) in enumerate([('OLD', OLD), ('NEW', NEW)]):
        m = get_mask(b0['image'], R=R, mode='uniform', offset=0, **p)
        n, acs = lines_of(m), resolve_acs_lines(W, p['acs_lines'], p['center_frac'])
        row += [n, W / n, acs]
        ax[i, j].imshow(m[0, 0].cpu(), cmap='gray', aspect='auto', interpolation='nearest')
        ax[i, j].set_title(f"{name}  R={R}: {n} lines, A_eff {W / n:.1f}", fontsize=9)
        ax[i, j].axis('off')
    print(f"{R:>3} | {row[0]:>9} {row[1]:>6.2f} {row[2]:>4} | {row[3]:>9} {row[4]:>6.2f} {row[5]:>4}")
plt.tight_layout(); plt.show()

## 2) The scanner's own noise

`kspace_awgn` adds noise to *measured* k-space, so the total is
$\sqrt{\sigma_0^2 + \sigma^2}$ and `sigma = 0` is not noiseless. $\sigma_0$ is read
the classic way: RMS of the fully sampled coil images over air, pushed clear of
the object (`OBJ_THRESH`, `DILATE`). `fftc` is orthonormal and `_randn_like` has
$E|z|^2 = 1$, so this RMS is on the same scale as `sigma`.

On the phantom the true $\sigma_0$ is 0.004; the estimate should land on it.

In [ ]:
OBJ_THRESH, DILATE = 0.05, 8

s0 = []
for b in batches:
    coil = ifftc(b['kspace'])
    rss = coil.abs().pow(2).sum(1, keepdim=True).sqrt()
    obj = (rss > OBJ_THRESH * rss.amax()).float()
    k = 2 * DILATE + 1
    air = torch.nn.functional.max_pool2d(obj, k, stride=1, padding=k // 2) < 0.5
    s0.append(float(coil[air.expand_as(coil)].abs().pow(2).mean().sqrt())
              if air.any() else float('nan'))
SIGMA_0 = float(np.nanmean(s0))
print(f"sigma_0 = {SIGMA_0:.4f}   per slice: " + ", ".join(f"{v:.4f}" for v in s0))
print(f"air fraction (first slice) {air.float().mean():.2f}\n")
print(f"{'sigma':>7}  {'total sqrt(s0^2+s^2)':>21}")
for s in SIGMA_GRID:
    print(f"{s:>7.3f}  {math.hypot(SIGMA_0, s):>21.4f}")

## 3) The sweep

For every slice, R and sigma:

* **NEW** -- `measurement_awgn` + online `SMAP_METHOD` maps from the measurement's
  own ACS, exactly as training builds it.
* **ORACLE** -- the same NEW measurement, reconstructed with the **stored** maps.
  Not something training can do (those maps define the ground truth); it is
  here to split the difficulty. NEW vs ORACLE is what the online maps cost;
  ORACLE vs OLD is what the mask costs.
* **OLD** -- `simulated` + the stored maps, 20 ACS lines, nominal R: what the old
  noise level was chosen on.

Online maps cost more than their accuracy suggests, for two reasons. They are
calibrated from 13 lines, so they are smooth along phase-encode exactly where
the unfolding needs them sharp. And the stored ESPIRiT maps are thresholded:
exactly zero outside the object, so a SENSE solve with them never has to
estimate the background -- a support prior the old protocol got for free, and
online maps (`thresh_eig=0`) do not have. ORACLE carries that prior too.

then CG-SENSE with the operator's maps. The noise draw is seeded once (`SEED`)
and only scaled across the grid, so neighbouring columns differ in sigma alone.

In [ ]:
def measure(b, R, proto, sigma, method=None, legacy=False):
    mask = get_mask(b['image'], R=R, mode='uniform', offset=0, **proto)
    acs = resolve_acs_lines(b['image'].shape[-1], proto['acs_lines'], proto['center_frac'])
    g = torch.Generator(device=device).manual_seed(SEED)
    y, _, extra = prepare_measurement(
        image=b['image'], kspace=b['kspace'], mask=mask, smaps=b['smaps'],
        kspace_type='simulated' if legacy else 'measurement_awgn',
        noise_std=float(sigma), noise_dist='uniform', whiten_kspace=False,
        generator=g, online_smaps=None if legacy else (method or SMAP_METHOD),
        online_smaps_kws=dict(acs_lines=acs))
    return y, mask, extra['smaps']


@torch.no_grad()
def sense(y, mask, smaps):
    E = Mask(mask) @ FFT2D() @ Sense(smaps)
    x, _ = cg(lambda v: E.H(E(v)) + LAMDA * v, E.H(y), max_iter=CG_ITERS, tol=CG_TOL)
    return x


def scores(b, x):
    gt, xr = b['image'].abs(), x.abs()
    m = compute_metrics(gt, xr, mask=b['organ'])
    u = compute_metrics(gt, xr)
    return {**{k: float(v) for k, v in m.items()},
            **{k + '_all': float(v) for k, v in u.items()}}


old_grid = sorted(set(SIGMA_GRID) | {OLD_SIGMA})
results, recons = {}, {}
for tag, proto, legacy, grid in [('NEW', NEW, False, SIGMA_GRID), ('OLD', OLD, True, old_grid)]:
    for R in R_LIST:
        for s in grid:
            rows, rows_or = [], []
            for i, b in enumerate(batches):
                y, mask, smaps = measure(b, R, proto, s, legacy=legacy)
                x = sense(y, mask, smaps)
                rows.append(scores(b, x))
                if tag == 'NEW':                        # same y, stored maps
                    rows_or.append(scores(b, sense(y, mask, b['smaps'])))
                if i == 0:
                    recons[(tag, R, s)] = x[0, 0].abs().cpu()
            results[(tag, R, s)] = {k: float(np.mean([r[k] for r in rows])) for k in rows[0]}
            if rows_or:
                results[('ORACLE', R, s)] = {k: float(np.mean([r[k] for r in rows_or]))
                                             for k in rows_or[0]}
        print(f"{tag} R={R:<2} done: NRMSE " + " ".join(
            f"{results[(tag, R, s)]['nrmse']:.3f}" for s in grid))

### Reconstructions (first slice)

Rows are R, columns sigma, all under the NEW protocol. The first column is the
ground truth, the second the OLD protocol at its old operating point
`OLD_SIGMA` -- the picture the current noise level was chosen from. One display
window for every panel (`VMAX`, the ground truth's max), so brightness is
comparable across the whole grid. Titles give masked NRMSE / SSIM.

In [ ]:
ncol = len(SIGMA_GRID) + 2
fig, ax = plt.subplots(len(R_LIST), ncol, figsize=(2.1 * ncol, 2.5 * len(R_LIST)),
                       squeeze=False)
gt0 = b0['image'][0, 0].abs().cpu()
for i, R in enumerate(R_LIST):
    old = results[('OLD', R, OLD_SIGMA)]
    panels = [(gt0, 'ground truth'),
              (recons[('OLD', R, OLD_SIGMA)],
               f"OLD s={OLD_SIGMA}\n{old['nrmse']:.3f} / {old['ssim']:.3f}")]
    for s in SIGMA_GRID:
        r = results[('NEW', R, s)]
        panels.append((recons[('NEW', R, s)], f"s={s}\n{r['nrmse']:.3f} / {r['ssim']:.3f}"))
    for j, (img, ttl) in enumerate(panels):
        ax[i, j].imshow(img, cmap='gray', vmin=0, vmax=VMAX)
        ax[i, j].set_title(ttl, fontsize=8)
        ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
    ax[i, 0].set_ylabel(f"R={R}", fontsize=10)
fig.suptitle(f"CG-SENSE ({CG_ITERS} it), NEW protocol, operator maps: {SMAP_METHOD}", y=1.0)
plt.tight_layout(); plt.show()

### Difficulty vs sigma, and the matched level

Solid: NEW. Dash-dot: ORACLE (NEW measurement, stored maps). Dashed: OLD.
The dotted line in each panel is OLD at `OLD_SIGMA` --
the difficulty the old training range was chosen to have. The table reads off
the NEW sigma with the same masked NRMSE at the same nominal R.

If NEW at **sigma = 0** already sits above that line, no amount of noise
reduction gives the old difficulty back: the undersampling and the online maps
alone are harder than the old protocol at 0.05. That is a statement about R,
not about sigma.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for c, R in enumerate(R_LIST):
    col = f"C{c}"
    for k, a in zip(('nrmse', 'ssim'), ax):
        a.plot(SIGMA_GRID, [results[('NEW', R, s)][k] for s in SIGMA_GRID], 'o-',
               color=col, label=f"NEW R={R}")
        a.plot(SIGMA_GRID, [results[('ORACLE', R, s)][k] for s in SIGMA_GRID], '^-.',
               color=col, alpha=.8, label=f"ORACLE R={R}")
        a.plot(old_grid, [results[('OLD', R, s)][k] for s in old_grid], 's--',
               color=col, alpha=.6, label=f"OLD R={R}")
        a.axhline(results[('OLD', R, OLD_SIGMA)][k], color=col, ls=':', lw=1)
for a, k in zip(ax, ('NRMSE (masked)', 'SSIM (masked)')):
    a.set_xlabel('added sigma'); a.set_ylabel(k); a.grid(alpha=.3)
ax[0].legend(fontsize=7, ncol=3)
plt.tight_layout(); plt.show()

print(f"NEW-protocol sigma matching OLD at sigma={OLD_SIGMA} (masked NRMSE, same nominal R)")
print(f"{'R':>3} {'OLD nrmse':>10} {'NEW @ s=0':>10} {'ORACLE @ s=0':>13} {'matched sigma':>14}")
for R in R_LIST:
    target = results[('OLD', R, OLD_SIGMA)]['nrmse']
    curve = np.array([results[('NEW', R, s)]['nrmse'] for s in SIGMA_GRID])
    if target < curve[0]:
        msg = 'none: harder at s=0'
    elif target > curve.max():
        msg = f'> {max(SIGMA_GRID)}'
    else:
        msg = f"{np.interp(target, np.maximum.accumulate(curve), SIGMA_GRID):.4f}"
    print(f"{R:>3} {target:>10.4f} {curve[0]:>10.4f} "
          f"{results[('ORACLE', R, SIGMA_GRID[0])]['nrmse']:>13.4f} {msg:>14}")

print("\nfull table (masked | unmasked NRMSE, masked SSIM)")
for R in R_LIST:
    print(f"R={R}: " + "  ".join(
        f"{s}:{results[('NEW', R, s)]['nrmse']:.3f}|{results[('NEW', R, s)]['nrmse_all']:.3f}"
        f"/{results[('NEW', R, s)]['ssim']:.3f}" for s in SIGMA_GRID))

## 4) Coil maps from the undersampled, noisy measurement

At `SHOW_R`, `SHOW_SIGMA`: the maps `online_smaps` estimates from the 13-line ACS
of the noisy measurement, against the stored ESPIRiT maps (estimated from the
fully sampled data -- they define the ground truth, and never reach the operator
under the measured protocol).

Maps are only defined up to a per-pixel phase shared by all coils, so the online
maps are aligned to the stored ones before comparison:
$\phi = \arg\sum_c \bar s^\text{stored}_c s_c$, $s_c \leftarrow s_c e^{-i\phi}$.
The error map is $\lVert s - s^\text{stored} \rVert_2$ over coils, on the stored
support. Two summary numbers per estimator:

* **map err** -- mean of that error over the support (0 = identical, 1.41 =
  orthogonal);
* **combine NRMSE** -- the fully sampled coils combined with these maps,
  $\sum_c \bar s_c m_c / \sum_c |s_c|^2$, against the ground truth. This is the
  error the maps alone cost, with no undersampling in the way.

In [ ]:
y_s, mask_s, _ = measure(b0, SHOW_R, NEW, SHOW_SIGMA)
acs_s = resolve_acs_lines(W, NEW['acs_lines'], NEW['center_frac'])
maps = {'stored': b0['smaps']}
with torch.no_grad():
    maps['walsh'] = online_smaps(y_s, mask_s, method='walsh', acs_lines=acs_s)
    if INCLUDE_ESPIRIT:
        try:
            maps['espirit'] = online_smaps(y_s, mask_s, method='espirit', acs_lines=acs_s)
        except RuntimeError as ex:                      # OOM at full brain grid
            print(f"online ESPIRiT skipped: {str(ex).splitlines()[0][:100]}")
            if device == 'cuda':
                torch.cuda.empty_cache()


def align(s, ref):
    ph = torch.angle((ref.conj() * s).sum(1, keepdim=True))
    return s * torch.exp(-1j * ph)


sup = b0['organ']
coil_full = ifftc(b0['kspace'])
print(f"R={SHOW_R}  sigma={SHOW_SIGMA}  ACS lines={acs_s}  lines kept={lines_of(mask_s)}\n")
print(f"{'maps':<9}{'map err':>9}{'combine NRMSE':>15}{'RSS on support':>16}")
for name, s in maps.items():
    sa = align(s, maps['stored']) if name != 'stored' else s
    err = (sa - maps['stored']).abs().pow(2).sum(1, keepdim=True).sqrt()
    comb = (s.conj() * coil_full).sum(1, keepdim=True) / \
        s.abs().pow(2).sum(1, keepdim=True).clamp_min(1e-8)
    nr = compute_metrics(b0['image'].abs(), comb.abs(), mask=sup)['nrmse']
    rss = s.abs().pow(2).sum(1).sqrt()[sup[:, 0]]
    print(f"{name:<9}{err[sup].mean():>9.3f}{float(nr):>15.4f}{rss.mean():>10.3f} +- {rss.std():.3f}")
    maps[name] = sa

In [ ]:
names = list(maps)
nc = min(N_COILS_SHOW, C)
for part, cmap, lim in [('magnitude', 'viridis', (0, 1)), ('phase', 'twilight', (-math.pi, math.pi))]:
    fig, ax = plt.subplots(len(names), nc, figsize=(1.9 * nc, 2.0 * len(names)), squeeze=False)
    for i, n in enumerate(names):
        for c in range(nc):
            s = maps[n][0, c]
            img = s.abs() if part == 'magnitude' else torch.where(sup[0, 0], s.angle(), 0.0)
            ax[i, c].imshow(img.cpu(), cmap=cmap, vmin=lim[0], vmax=lim[1])
            ax[i, c].set_xticks([]); ax[i, c].set_yticks([])
            if i == 0:
                ax[i, c].set_title(f"coil {c}", fontsize=8)
        ax[i, 0].set_ylabel(n, fontsize=9)
    fig.suptitle(f"coil maps, {part}" + (" (phase-aligned to stored, on support)"
                                         if part == 'phase' else ""), y=1.0)
    plt.tight_layout(); plt.show()

fig, ax = plt.subplots(2, len(names), figsize=(3.2 * len(names), 6.2), squeeze=False)
for j, n in enumerate(names):
    rss = maps[n][0].abs().pow(2).sum(0).sqrt().cpu()
    err = (maps[n] - maps['stored'])[0].abs().pow(2).sum(0).sqrt() * sup[0, 0]
    im = ax[0, j].imshow(rss, cmap='magma', vmin=0, vmax=1.2); ax[0, j].set_title(f"{n}: coil RSS", fontsize=9)
    plt.colorbar(im, ax=ax[0, j], fraction=.046)
    im = ax[1, j].imshow(err.cpu(), cmap='inferno', vmin=0, vmax=1.0); ax[1, j].set_title(f"{n}: |s - s_stored|", fontsize=9)
    plt.colorbar(im, ax=ax[1, j], fraction=.046)
for a in ax.ravel():
    a.axis('off')
plt.tight_layout(); plt.show()

## 5) The coil images behind the maps

Same slice, R and sigma. Four rows per coil:

1. **fully sampled** -- `ifftc(kspace)`, the measured coil images (scanner noise
   included);
2. **calibration** -- `ifftc(hamming(cm . y))`: the 13-line ACS of the noisy
   measurement, windowed. *This is all Walsh ever sees*, which is why its maps
   are blurred along phase-encode;
3. **aliased** -- `ifftc(y)`, the zero-filled undersampled measurement;
4. **model** -- `s_c . x_gt` with the online `SMAP_METHOD` maps: what the
   operator believes coil `c` looks like. Differences from row 1 are map error
   the net has to absorb.

Rows 1, 3, 4 share one window (the fully sampled coils' max). Row 2 has its own:
windowing and truncation shrink it, and on row 1's scale it would be black.

In [ ]:
lines_acs = acs_s
cm = center_mask(mask_s, lines_acs)
calib = ifftc(hamming_window(y_s * cm.to(y_s.dtype)))
rows = [('fully sampled', coil_full), ('calibration', calib), ('aliased', ifftc(y_s)),
        (f"model ({SMAP_METHOD})", maps[SMAP_METHOD] * b0['image'])]
v_full = float(coil_full.abs().max())
v_cal = float(calib.abs().max())
fig, ax = plt.subplots(len(rows), nc, figsize=(1.9 * nc, 2.0 * len(rows)), squeeze=False)
for i, (n, t) in enumerate(rows):
    vm = v_cal if n == 'calibration' else v_full
    for c in range(nc):
        ax[i, c].imshow(t[0, c].abs().cpu(), cmap='gray', vmin=0, vmax=vm)
        ax[i, c].set_xticks([]); ax[i, c].set_yticks([])
        if i == 0:
            ax[i, c].set_title(f"coil {c}", fontsize=8)
    ax[i, 0].set_ylabel(n, fontsize=9)
fig.suptitle(f"coil images, R={SHOW_R}, sigma={SHOW_SIGMA}, ACS={lines_acs} lines", y=1.0)
plt.tight_layout(); plt.show()

## Choosing the level

* **Visual criterion (the usual one):** from the reconstruction grid, the
  largest sigma at which the NEW recon is still recognizable but fairly noisy --
  per R, since R=4 and R=16 will not agree.
* **Where the difficulty comes from:** NEW vs ORACLE in section 3. A large
  gap at small sigma is map error, and no noise setting removes it -- that is
  what the Walsh switch (and its ACS size) acts on.
* **Difficulty-matched:** the table in section 3. If R=16 reads "harder at s=0",
  the NEW R=16 arm is harder than anything the old grid trained on regardless of
  sigma, and the choice is between a much lower sigma and a less aggressive R.
* **The floor:** anything below `sigma_0` (section 2) changes the total noise
  very little -- `sqrt(sigma_0^2 + sigma^2)` is dominated by the scanner there.

The range lives in `scripts/make_mg_recon_configs.py::NOISE_STD["brain"]` (val is
derived as its mean). Changing it makes the launch guard in
`torch/_mg_recon_body.sh` refuse existing brain run dirs, by design: move them
aside or pass `FORCE_RESTART=1`.